# OpticIQ, make a test volume for the app

Saves every slice of one OCT volume, one eye at one visit, as a zip on Drive, so the volume page of the app can be tried.
Only test patients are offered, so the model never saw the volume in training. The true labels per slice are saved next to the zip.

Reads the files the training notebook already made, and changes nothing in them
* `MyDrive/ClinIQ/OLIVES_Dataset_Labels.zip`, the labels
* `MyDrive/ClinIQ/olives_labeled_224.zip`, the cached slices
* `MyDrive/ClinIQ/common/splits.csv`, the shared patient split

Writes to `MyDrive/ClinIQ/sample_volumes`. A GPU is not needed.

In [ ]:
# setup and paths
import os, re, io, zipfile
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ClinIQ"
LABELS_ZIP = f"{DRIVE_DIR}/OLIVES_Dataset_Labels.zip"
CACHE_ZIP = f"{DRIVE_DIR}/olives_labeled_224.zip"
SPLIT_CSV = f"{DRIVE_DIR}/common/splits.csv"
OUT_DIR = f"{DRIVE_DIR}/sample_volumes"
os.makedirs(OUT_DIR, exist_ok=True)
for p in [LABELS_ZIP, CACHE_ZIP, SPLIT_CSV]:
    print(os.path.exists(p), p)

In [ ]:
# the labels, read straight from the zip, in the same row order the cache was built in
with zipfile.ZipFile(LABELS_ZIP) as z:
    csv_name = next(n for n in z.namelist() if n.endswith("full_labels/Biomarker_Clinical_Data_Images.csv"))
    df = pd.read_csv(io.BytesIO(z.read(csv_name)))

PATH_COL = df.columns[0]
LABELS = list(df.columns[2:18])
df[LABELS] = df[LABELS].fillna(0).astype(int)
df["row"] = range(len(df))                                     # cache image number, 00000.png and so on

# the same patient id and split as the training notebook
df["trial"] = df[PATH_COL].str.split("/").str[1]
df["pid"] = df["trial"] + "_" + df["Patient_ID"].astype(str)
df = df.merge(pd.read_csv(SPLIT_CSV), on="pid")
df["volume"] = df[PATH_COL].map(os.path.dirname)               # slices in the same folder are one volume
print(len(df), "slices,", df["pid"].nunique(), "patients,", df["volume"].nunique(), "volumes")

In [ ]:
# complete test volumes, the ones with the most biomarkers first
test = df[df["fold"] == -1]
sizes = test.groupby("volume").size()
full = sizes[sizes == 49].index
print(len(full), "complete test volumes")
print()

info = test[test["volume"].isin(full)].groupby("volume")[LABELS].max()
info["n_biomarkers"] = info[LABELS].sum(axis=1)
info = info.sort_values("n_biomarkers", ascending=False)
for k, (v, r) in enumerate(info.iterrows()):
    print(k, v, "|", ", ".join(l for l in LABELS if r[l] == 1) or "none")

In [ ]:
PICK = 0                                                       # change to choose another volume from the list above
volume = info.index[PICK]

# slices in order by the Scan (n/49) column, the slice position in the volume
def scan_number(v):
    numbers = re.findall(r"\d+", str(v))
    return int(numbers[0]) if numbers else 0
rows = test[test["volume"] == volume].copy()
rows = rows.assign(n=rows["Scan (n/49)"].map(scan_number)).sort_values(["n", PATH_COL])

# copy just these slices out of the cache zip into a new zip, numbered in order
name = volume.strip("/").replace("/", "_").replace(" ", "")
zip_path = f"{OUT_DIR}/{name}.zip"
with zipfile.ZipFile(CACHE_ZIP) as cache, zipfile.ZipFile(zip_path, "w") as out:
    in_cache = {os.path.basename(n): n for n in cache.namelist()}
    for k, row in enumerate(rows["row"], start=1):
        out.writestr(f"slice_{k:02d}.png", cache.read(in_cache[f"{row:05d}.png"]))

# the true labels per slice, to compare with what the app finds
rows.assign(slice=range(1, len(rows) + 1))[["slice", PATH_COL] + LABELS].to_csv(f"{OUT_DIR}/{name}_labels.csv", index=False)
print("saved", zip_path, "with", len(rows), "slices")
print("present somewhere in this volume |", ", ".join(l for l in LABELS if rows[l].max() == 1) or "none")

Download the zip from `MyDrive/ClinIQ/sample_volumes` to your computer, then upload it on the volume page of the app. Compare its Findings tab with the `_labels.csv` saved next to the zip.

## Volume level results on all 40 test volumes

Runs the fold 3 model from Hugging Face on every test slice, then scores it two ways
* slice level, each slice against its own labels, the way the paper reports it, should come out near 0.449
* volume level, a biomarker counts as present in a volume when at least `MIN_SLICES` slices cross its threshold, scored against whether any slice in that volume has it

F1 is computed the same way as the training notebook. Biomarkers with no positives get no score and are skipped in the average, and the headline uses only biomarkers with at least 3 test patients.
Needs the cells above run first and the Colab secret `CLINIQ_HF_TOKEN`. A GPU makes it take a couple of minutes, on CPU it takes about 10.

In [ ]:
# load the published model and its thresholds, the same ones the app uses
!pip -q install timm huggingface_hub
import json, numpy as np, torch, timm
import torchvision.transforms as T
from PIL import Image
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

REPO_ID = "cliniq2027/cliniq-olives-vit-base-224-16bio"
login(token=userdata.get("CLINIQ_HF_TOKEN"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model("hf-hub:" + REPO_ID, pretrained=True).to(DEVICE).eval()
THRESHOLDS = json.load(open(hf_hub_download(REPO_ID, "thresholds.json")))
assert list(model.pretrained_cfg["label_names"]) == LABELS, "label order differs from the CSV"
thresholds = np.array([float(THRESHOLDS[name]) for name in LABELS])

# the same preparation as training and the app
prepare = T.Compose([T.Resize((224, 224)), T.Grayscale(3), T.ToTensor(),
                     T.Normalize(model.pretrained_cfg["mean"], model.pretrained_cfg["std"])])
print("model ready on", DEVICE)

In [ ]:
# probabilities for every test slice, read straight from the cache zip
test = df[df["fold"] == -1].reset_index(drop=True)
probs = []
with zipfile.ZipFile(CACHE_ZIP) as cache:
    in_cache = {os.path.basename(n): n for n in cache.namelist()}
    for i in range(0, len(test), 64):
        batch = [prepare(Image.open(io.BytesIO(cache.read(in_cache[f"{r:05d}.png"]))).convert("L"))
                 for r in test["row"].iloc[i:i + 64]]
        with torch.no_grad():
            probs.append(torch.sigmoid(model(torch.stack(batch).to(DEVICE))).cpu().numpy())
probs = np.concatenate(probs)
truth = test[LABELS].values
print(probs.shape[0], "test slices from", test["pid"].nunique(), "patients and", test["volume"].nunique(), "volumes")

In [ ]:
# F1 per biomarker, the same as biomarker_f1 in the training notebook
def biomarker_f1(predicted, present):
    tp = (predicted & present).sum(axis=0)
    fp = (predicted & ~present).sum(axis=0)
    fn = (~predicted & present).sum(axis=0)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / np.maximum(tp + fn, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-9)
    f1[present.sum(axis=0) == 0] = np.nan          # no positives, no meaningful F1
    return f1

# biomarkers with at least 3 test patients, the same headline set as the paper
patients_with = np.array([test.loc[test[name] == 1, "pid"].nunique() for name in LABELS])
measurable = patients_with >= 3

# slice level
slice_f1 = biomarker_f1(probs >= thresholds, truth == 1)
print(f"slice level    macro F1 all {np.nanmean(slice_f1):.3f}   over the {measurable.sum()} with 3 or more test patients {np.nanmean(slice_f1[measurable]):.3f}")
print()

# volume level, one row per volume
order = test["volume"].values
volumes = pd.unique(order)
volume_truth = np.array([truth[order == v].max(axis=0) for v in volumes]) == 1
slices_above = np.array([(probs[order == v] >= thresholds).sum(axis=0) for v in volumes])

volume_f1 = {}
for m in [1, 2, 3, 5]:
    volume_f1[m] = biomarker_f1(slices_above >= m, volume_truth)
    print(f"volume level, present on at least {m} slices   macro F1 all {np.nanmean(volume_f1[m]):.3f}   "
          f"over the {measurable.sum()} {np.nanmean(volume_f1[m][measurable]):.3f}")

In [ ]:
# per biomarker, slice level next to volume level with the app setting of 2 slices
table = pd.DataFrame({
    "biomarker": LABELS,
    "test patients": patients_with,
    "volumes with it": volume_truth.sum(axis=0),
    "slice F1": slice_f1,
    "volume F1, 2 slices": volume_f1[2],
    "volumes called present": (slices_above >= 2).sum(axis=0),
}).sort_values("test patients", ascending=False)
table.round(3)